In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score

# -------------------------------
# (F)(G) 設定
# -------------------------------
BASE_PATH = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
TARGET_VARS = ["Jsc", "Voc", "FF", "PCEmax"]

# (D)(E) 除外変数設定
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

FILE_NAMES = [
    "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
    "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
    "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
    "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
]

OUT_ROOT = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
MODEL_NAME = "MLP"
RUN_DIR = os.path.join(OUT_ROOT, f"Corr_1000_{MODEL_NAME}")
os.makedirs(RUN_DIR, exist_ok=True)

# -------------------------------
# HELPERS
# -------------------------------
def safe_r2(y_true, y_pred):
    try: return float(r2_score(y_true, y_pred))
    except: return np.nan

def remove_collinear_features(df, cols, threshold=0.99999):
    if len(cols) < 2: return cols
    corr = df[cols].corr().abs().fillna(0.0)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    return [c for c in cols if c not in to_drop]

def calc_permutation_importance(model, X, y, features, base_r2):
    importances = {}
    X_orig = X.copy()
    for i, col in enumerate(features):
        X_perm = X_orig.copy()
        np.random.shuffle(X_perm[:, i])
        y_pred_perm = model.predict(X_perm)
        new_r2 = safe_r2(y, y_pred_perm)
        importances[col] = max(0, base_r2 - (new_r2 if np.isfinite(new_r2) else 0))
    return importances

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list = []
importance_list = []

for file_name in FILE_NAMES:
    file_path = os.path.join(BASE_PATH, file_name)
    if not os.path.exists(file_path): continue

    print(f"\n=== Processing MLP: {file_name} ===")
    df_raw = pd.read_csv(file_path, index_col=0).drop(columns=["X"], errors="ignore")
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()
    if len(df_num) < 20: continue

    for target in TARGET_VARS:
        if target not in df_num.columns: continue

        # (C)(D)(E) 変数選択
        feature_cols = [c for c in df_num.columns if c not in TARGET_VARS]
        feature_cols = [c for c in feature_cols if c not in INAPPROPRIATE_VARS]
        for pat in PHYSICAL_EXCLUDE_PATTERNS:
            feature_cols = [c for c in feature_cols if pat not in c]

        # (H) 多重共線性・ゼロ分散除外
        v = df_num[feature_cols].var()
        feature_cols = v[v > 0].index.tolist()
        feature_cols = remove_collinear_features(df_num, feature_cols)
        if len(feature_cols) == 0: continue

        # (I) スケーリング [-1, 1]
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_scaled = scaler.fit_transform(df_num[feature_cols])
        y = df_num[target].to_numpy()

        # --- Hyperparameter tuning ---
        param_grid = {
            "hidden_layer_sizes": [(32,), (64,), (128,)],
            "alpha": [1e-4, 1e-3, 1e-2]
        }
        cv = KFold(n_splits=10, shuffle=True, random_state=42)
        grid = GridSearchCV(
            MLPRegressor(max_iter=2000, random_state=42),
            param_grid, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
        )
        grid.fit(X_scaled, y)
        best_model = grid.best_estimator_

        # (K) CV10 Out-of-Fold 予測
        y_oof = cross_val_predict(best_model, X_scaled, y, cv=cv)
        
        # 指標
        cv_r2 = safe_r2(y, y_oof)
        cv_rmse = np.sqrt(mean_squared_error(y, y_oof))

        # --- 保存処理 ---
        file_tag = os.path.splitext(file_name)[0]
        
        # 1. モデル保存
        joblib.dump(best_model, os.path.join(RUN_DIR, f"{file_tag}_{target}_model.pkl"))

        # 2. (B)(K) CV10_OOF 予測CSV
        df_oof = pd.DataFrame({
            "SampleID": df_num.index.astype(str),
            "Observed": y,
            "Predicted": y_oof,
            "Type": "CV10_OOF"
        })
        df_oof.to_csv(os.path.join(RUN_DIR, f"{file_tag}_{target}_pred_OOF.csv"), index=False)

        # 3. (J) 重要度 (Permutation)
        base_r2_fit = safe_r2(y, best_model.predict(X_scaled))
        imps = calc_permutation_importance(best_model, X_scaled, y, feature_cols, base_r2_fit)
        for f, val in imps.items():
            if val > 1e-6:
                importance_list.append({"File": file_name, "Target": target, "Feature": f, "Importance": val})

        # 4. サマリー
        summary_list.append({
            "File": file_name, "Target": target, "CV10_R2": cv_r2, "CV10_RMSE": cv_rmse,
            "n_samples": len(df_num), "n_features": len(feature_cols),
            "Best_Params": str(grid.best_params_)
        })
        print(f"  Target={target} | CV10_R2={cv_r2:.3f}")

# CSV保存
pd.DataFrame(summary_list).to_csv(os.path.join(RUN_DIR, "all_summary_CV10.csv"), index=False)
pd.DataFrame(importance_list).to_csv(os.path.join(RUN_DIR, "all_importance_MLP.csv"), index=False)
print("\n[DONE] MLP Analysis completed.")


=== Processing MLP: n_base.csv ===


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

  Target=Jsc | CV10_R2=0.574
  Target=Voc | CV10_R2=0.657
  Target=FF | CV10_R2=0.223


...

  Target=Jsc | CV10_R2=0.697
  Target=Voc | CV10_R2=0.363
  Target=FF | CV10_R2=-0.115
  Target=PCEmax | CV10_R2=0.616

=== Processing MLP: m_base.csv ===


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

  Target=Jsc | CV10_R2=0.205
  Target=Voc | CV10_R2=0.292
  Target=FF | CV10_R2=0.015


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Target=PCEmax | CV10_R2=0.230

=== Processing MLP: m_base_OH_rebuilt.csv ===
  Target=Jsc | CV10_R2=0.600
  Target=Voc | CV10_R2=0.313
  Target=FF | CV10_R2=0.078
